In [1]:
from services.ollama_client import OllamaConfig,OllamaClient
from dotenv import load_dotenv
from utils.reader import Reader
import os

In [2]:
load_dotenv()

config = OllamaConfig()

llm = OllamaClient(config)

projects_path = os.getenv('PROJECTS_PATH', 'projects')

project_data = Reader.read_readme(projects_path)

summarized_content = {}
for project in project_data:
    if project_data[project] == "No README.md" or not project_data[project]:
        print(f"No README.md found for {project} \n ")
        continue
        
    else:
        summary = llm.summaries_readme_llm_f(project_data[project],project)
        # print(f'the state after summary is {state["generation"]}')
        json = summary.model_dump()
        content = ''
        for key,value in json.items():
            content+= f'{key}: {value} \n'
        summarized_content[project] = content


print(f'the summary of my projects is: \n {summarized_content}')


No README.md found for Cash_flow_excel 
 
No README.md found for ROS2 
 
No README.md found for Train_LLM 
 
the summary of my projects is: 
 {'JobApp_Agent': 'description: Job Application Agent: An intelligent, automated agent designed to streamline the job application process. \ntech_stack: LangChain, LangGraph, Ollama (compatible with OpenAI API format), Python, Jupyter Notebook \nobjectives: Streamline job application process, assist users in managing applications, automate resume and cover letter generation, and provide a conversational interface. \nchallenges_solutions: Solves the challenges of manually updating resumes and cover letters for each job application, extracts key technical details from project READMEs, and generates optimized cover letters for ATS compatibility. \n', 'offside-detector': 'description: Create a system to detect when a player is in an offside position on the field. \ntech_stack: Python, OpenCV \nobjectives: Develop a system that can accurately detect wh

In [5]:
print(type(summary.model_dump()))

<class 'dict'>


In [3]:
# this is the test for jd keywords extractions
from services.ollama_client import OllamaConfig,OllamaClient
from dotenv import load_dotenv
from utils.reader import Reader
import os

load_dotenv()

config = OllamaConfig()

llm = OllamaClient(config)

query = "I am applying for this job https://www.worldquant.com/career-listing/?id=4408985006&source=c8fd37dd6us"

jd_content = await Reader.extract_jd(query)

jd_key_words = llm.jd_key_words_resume_llm_f(jd_content)

print(f'the jd key words are: \n {jd_key_words}')







URL pattern match is : https://www.worldquant.com/career-listing/?id=4408985006&source=c8fd37dd6us
the len of the HTML is 38724
the number of sections are: 12
the number of sections scored are  12
the best score is 2.0
the length of paragraph after splitting is 29
the number of paras appended in stop phrases is : 1
the length of paragraph after trimming by stop words is 29
the number of paras appended in density is : 24
the length of paragraph after splitting by density is 24
23540
the jd key words are: 
 job_title='Software Engineer' hard_skills='Python, AI, Language Models, Vector Databases, SQL, Redis, Kafka' soft_skills='Analytical Skills, Problem-Solving, Communication, Teamwork' tools_and_technologies='AI, LLMs, Vector Databases, SQL, Redis, Kafka' responsibilities='Design and build scalable AI-driven products addressing real-world problems' required_qualifications='Work experience as a software engineer with strong programming skills, preferably in Python' preferred_qualificatio

In [5]:
print(f"the keys are {jd_key_words.job_title}")
print(f"the resposibilities are {jd_key_words.responsibilities}")


the keys are Software Engineer
the resposibilities are Design and build scalable AI-driven products addressing real-world problems


In [4]:

load_dotenv()
query = "I am applying for this job https://www.worldquant.com/career-listing/?id=4408985006&source=c8fd37dd6us"

config = OllamaConfig()

llm = OllamaClient(config)

resume_content = Reader.get_resume()
print(f'the keys of the resume are {resume_content.keys()}')

jd_content = await Reader.extract_jd(query)

jd_key_words = llm.jd_key_words_resume_llm_f(jd_content)
jd_key_words = jd_key_words.model_dump_json()

print(f'the length of the jd is {len(jd_content)}')
print(f'the keys of the jd are {jd_key_words}')

updated_resume = {}

for resume_section, section_content in resume_content.items():
    # this is to ignore the header/Education section as this shouldnt be changed
    if resume_section == 'Header' or resume_section == 'Education':
        updated_resume[resume_section] = section_content
    else:
        response = llm.update_resume_llm_f(jd_key_words, str(summarized_content), section_content,resume_section)
        updated_resume[resume_section] = response.updated_content


# updated_resume = llm.update_resume_llm_f(jd_content['job_description'], summarized_content, resume_content)

print(f'the updated resume is: \n {updated_resume}')




Resume path is : ./data/resume.tex
Using local resume from ./data/resume.tex
Resume content: {'Header': 'Nikil PS\nCambridge,UK\n|\nnikilemuk@gmail.com\n|\n07871075988\n|\nLinkedin\n|\nGithhub', 'Summary': 'Software Engineer with 2 plus years of experience developing production-grade software across backend, LLMs, and embedded systems. Skilled in Python, C/C++, Docker, langchain, and pytorch, with a Master’s in Artificial Intelligence specializing in Computer Vision and ML model deployment', 'Skills and Certificates': 'Languages: Python, C, C++, JavaScript, SQL\nFrameworks/Libraries: Langchain, PyTorch, OpenCV, Scikit-learn, Pandas, HuggingFace, Langchain, FastAPI\nDomains: Computer Vision, NLP, Embedded Systems, Robotics\nTools: Docker, Kubernetes, Linux, AWS, SVN, VECTOR Toolset,Git,Ollama\nCertificates: NLP Specialization, CAN and CAN FD, Embedded C,Generative AI Specialisation', 'Experience': 'Software Engineer, Multimatic Electronic System -- Cambridge,GB\n•  Designed and containe

In [ ]:
from langgraph.graph import END, StateGraph, START
from IPython.display import display, Image
from graph.nodes import Nodes,GraphState
from graph.router import Router
from services.ollama_client import OllamaConfig
from typing_extensions import TypedDict
from typing import List
from pprint import pprint
import io
from PIL import Image
from dotenv import load_dotenv

load_dotenv()


/mnt/c/Users/nikil/Documents/Projects/JobApp_Agent/utils/latex_helper.py:197: SyntaxWarning: invalid escape sequence '\%'
  We use negative lookbehind to avoid touching already-escaped sequences like '\%'.


True

In [ ]:
def build_graph():

    state = GraphState()
    
    workflow = StateGraph(GraphState)
    workflow_nodes = Nodes(state)
    workflow_routers = Router(state)
    
    workflow.add_node("get_data", workflow_nodes.get_data)
    workflow.add_node("chat", workflow_nodes.chat)
    workflow.add_node("summarise_projects", workflow_nodes.summarise_projects)
    workflow.add_node("update_resume", workflow_nodes.update_resume)
    workflow.add_node("update_latex", workflow_nodes.update_latex) #updates and generates the latex file
    workflow.add_node("update_cover_letter", workflow_nodes.update_cover_letter)
    workflow.add_node("transcription_task", workflow_nodes.transcription_task)
    workflow.add_node("extract_jd_key_words_resume", workflow_nodes.extract_jd_keywords_resume)
    workflow.add_node("extract_jd_key_words_cl", workflow_nodes.extract_jd_keywords_cl)



    # Build the graph
    workflow.add_conditional_edges(
        START,
        workflow_routers.route_query,
        {
            "update_documents": "get_data",
            "internal_knowledge": "chat",
            "transcription": "transcription_task",
            "end": END,
        },
    )

    workflow.add_conditional_edges( 
        "chat",
        workflow_routers.route_query,
        {
            "internal_knowledge": END,
            "update_documents": END,
            "transcription": END,
            "end": END,
        },
    )

    workflow.add_edge("get_data", "summarise_projects")
    workflow.add_edge("summarise_projects","extract_jd_key_words_resume")
    workflow.add_edge("extract_jd_key_words_resume", "update_resume")
    workflow.add_edge("update_resume", "update_latex")
    workflow.add_edge("update_latex", "extract_jd_key_words_cl")
    workflow.add_edge("extract_jd_key_words_cl", "update_cover_letter")
    workflow.add_edge("transcription_task", END)

    # Compile
    app = workflow.compile()


    # debug to check the flow of the agent
    graph_png_bytes = app.get_graph().draw_mermaid_png()
    img = Image.open(io.BytesIO(graph_png_bytes))
    img.show()
    

    return app

In [ ]:
agent = build_graph()
# "I am applying for this job https://www.worldquant.com/career-listing/?id=4408985006&source=c8fd37dd6us"
inputs = {"query": "I am applying for this job https://www.worldquant.com/career-listing/?id=4408985006&source=c8fd37dd6us"}
async for output in agent.astream(inputs):
    for key, value in output.items():
        # Node
        pprint(f"Node '{key}':")
        # Optional: print full state at each node
        # pprint.pprint(value["keys"], indent=2, width=80, depth=None)
    pprint("\n---\n")

# Final generation
pprint(value.keys())

---ROUTE QUESTION---
---ROUTE QUESTION TO UPDATE CV,COVER LETTER---
---GET DATA---
Resume path is : ./data/resume.tex
Using local resume from ./data/resume.tex
Resume content: {'Header': 'Nikil PS\nCambridge,UK\n|\nnikilemuk@gmail.com\n|\n07871075988\n|\nLinkedin\n|\nGithhub', 'Summary': 'Software Engineer with 2 plus years of experience developing production-grade software across backend, LLMs, and embedded systems. Skilled in Python, C/C++, Docker, langchain, and pytorch, with a Master’s in Artificial Intelligence specializing in Computer Vision and ML model deployment', 'Skills and Certificates': 'Languages: Python, C, C++, JavaScript, SQL\nFrameworks/Libraries: Langchain, PyTorch, OpenCV, Scikit-learn, Pandas, HuggingFace, Langchain, FastAPI\nDomains: Computer Vision, NLP, Embedded Systems, Robotics\nTools: Docker, Kubernetes, Linux, AWS, SVN, VECTOR Toolset,Git,Ollama\nCertificates: NLP Specialization, CAN and CAN FD, Embedded C,Generative AI Specialisation', 'Experience': 'Softwa